In [1]:
# for package auto reload
%load_ext autoreload
%autoreload 2

# for better rendering of plots in jupyter notebook
%matplotlib inline

In [ ]:
# base modules
from pathlib import Path

# base modules
from collections import defaultdict

# for manipulating data
import numpy as np
import pandas as pd


In [ ]:
from pytrends.request import TrendReq
import time
import random

In [4]:
path_to_repo = Path('..').resolve()
path_to_data = path_to_repo / 'data'

In [5]:
df_raw = pd.read_csv(path_to_data / 'vestiaire.csv', low_memory = False)

In [6]:
df_raw = df_raw.convert_dtypes()

In [7]:
df_raw

,product_id,product_type,product_name,product_description,product_keywords,product_gender_target,product_category,product_season,product_condition,product_like_count,...,warehouse_name,seller_id,seller_username,usually_ships_within,seller_country,seller_products_sold,seller_num_products_listed,seller_community_rank,seller_num_followers,seller_pass_rate
0,43247626,Wool mini skirt,Wool mini skirt Miu Miu Grey size S Internatio...,Miu Miu – Pleated mini skirt Size: 36 (S) Wai...,Miu Miu Wool Skirts,Women,Women Clothing,Autumn / Winter,Never worn,34,...,Tourcoing,25775970,vitalii25775970,<NA>,Germany,3,14,0,13,0.0
1,43247441,Jacket,Jacket Barbara Bui Navy size 42 FR in Cotton,For selling nice women's suit Barbara Bui size...,Barbara Bui Cotton Jackets,Women,Women Clothing,All seasons,Very good condition,1,...,Tourcoing,13698770,olivia13698770,<NA>,Belgium,0,0,0,8,0.0
2,43246517,Wool coat,Wool coat Comme Des Garcons White size S Inter...,Magnificent boiled wool coat. I bought it in t...,Comme Des Garcons Wool Coats,Women,Women Clothing,Autumn / Winter,Very good condition,2,...,Tourcoing,6042365,cecilia6042365,1-2 days,Spain,58,69,0,62,96.0
3,43246507,Mini skirt,Mini skirt MSGM Black size 38 IT in Polyester,MSGM Skirt Black Printed Raw-Edge & Embroidere...,MSGM Polyester Skirts,Women,Women Clothing,All seasons,Very good condition,0,...,Brooklyn,13172949,gretchen13172949,1-2 days,United States,63,274,126346,131,96.0
4,43246417,Vegan leather trousers,Vegan leather trousers LVIR Black size 36 FR i...,LVIR black grained faux leather trousers size ...,LVIR Vegan leather Trousers,Women,Women Clothing,All seasons,Very good condition,1,...,Crawley,2578605,crunchykat,3-5 days,United Kingdom,19,14,102821,40,89.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
900509,41538360,Glitter trainers,Glitter trainers Bally Gold size 38 EU in Glit...,"Bally Vita Parcours sneakers. PPleather, suede...",Bally Glitter Trainers,Women,Women Shoes,All seasons,Good condition,1,...,Tourcoing,8849230,lauragua,1-2 days,Italy,40,17,391778,104,100.0
900510,41532271,Leather heels,Leather heels Casadei Black size 38.5 EU in Le...,Trendy and classic Casadei high-heel pumps Mad...,Casadei Leather Heels,Women,Women Shoes,All seasons,Good condition,1,...,Tourcoing,5448248,bettina5448248,1-2 days,Austria,26,99,51408,75,89.0
900511,41538140,Leather cowboy boots,Leather cowboy boots Ash Black size 36 EU in L...,Very good quality leather boots Worn once Elas...,Ash Leather Boots,Women,Women Shoes,All seasons,Very good condition,0,...,Tourcoing,9347694,sylvie9347694,<NA>,France,0,2,0,3,0.0
900512,41537603,Leather ballet flats,Leather ballet flats Lauren Ralph Lauren Black...,Very beautiful ballet flats like new. I don't ...,Lauren Ralph Lauren Leather Ballet flats,Women,Women Shoes,All seasons,Very good condition,27,...,Tourcoing,24074881,marina24074881,1-2 days,Italy,2,7,0,11,100.0


Sort brand names by order of counts. 

In [9]:
brand_counts = df_raw['brand_name'].value_counts().reset_index()

brand_counts

,brand_name,count
0,Gucci,41009
1,Burberry,24018
2,Dolce & Gabbana,22024
3,Prada,20972
4,Hermès,18711
...,...,...
8879,Carla Montanarini,1
8880,Pako Litto,1
8881,Themoirè,1
8882,Le Capsole,1


Extract and brand names as a list.

In [10]:
brands_sorted = brand_counts["brand_name"].tolist()

brands_sorted

['Gucci',
 'Burberry',
 'Dolce & Gabbana',
 'Prada',
 'Hermès',
 'Louis Vuitton',
 'Chanel',
 'Nike',
 'Valentino Garavani',
 'Balenciaga',
 'Dior',
 'Fendi',
 'Adidas',
 'Saint Laurent',
 'Versace',
 'Polo Ralph Lauren',
 'Christian Louboutin',
 'Bottega Veneta',
 'Yves Saint Laurent',
 'Dsquared2',
 'Givenchy',
 'Salvatore Ferragamo',
 "Tod's",
 'Max Mara',
 'Celine',
 'Alexander McQueen',
 'Jimmy Choo',
 'Dior Homme',
 'Moncler',
 'Miu Miu',
 'Moschino',
 'Michael Kors',
 'D&G',
 'Sandro',
 'Boss',
 'Emporio Armani',
 'Tommy Hilfiger',
 'Acne Studios',
 'Kenzo',
 'Missoni',
 'Lanvin',
 'Giorgio Armani',
 'JORDAN',
 'Off-White',
 "Levi's",
 'Comme Des Garcons',
 'Diesel',
 'Tom Ford',
 'Loro Piana',
 'Balmain',
 'Lacoste',
 'GUESS',
 'Brunello Cucinelli',
 'Ray-Ban',
 'Roberto Cavalli',
 'Chloé',
 'Etro',
 'Ermenegildo Zegna',
 'Hogan',
 'Maje',
 'Cartier',
 "Church's",
 'Stone Island',
 'Maison Martin Margiela',
 'Ralph Lauren',
 'Giuseppe Zanotti',
 'Gianni Versace',
 'Paul Smith',

Scraping Google Trends is notoriously tedious, we can expect to eventually get IP banned. We resume scraping from the last brand that was scraped and we randomly rotate between agents.

In [17]:
res = "google_trends_all_brands.csv"

try:
    existing = pd.read_csv(res)
    done_brands = set(existing["brand"].unique())
except FileNotFoundError:
    existing = pd.DataFrame()
    done_brands = set()

agents = [
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64)",
    "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7)",
    "Mozilla/5.0 (X11; Linux x86_64)",
]

Note that for brands with high counts, batch processing did not work.

We scraped worldwide, in the last 12 months and added random wait times if we encounter an error.

In [12]:
def fetch_trends_batch(brands, max_retries=5):
    """
    Fetch Google Trends data for a list of brands in a single batch.
    Returns a tidy DataFrame with columns: date, value, isPartial, brand.
    """
    for attempt in range(1, max_retries + 1):
        try:
            ua = random.choice(agents)
            pytrends = TrendReq(hl="en-US", tz=360, retries=0, backoff_factor=0)
            pytrends.headers["User-Agent"] = ua

            pytrends.build_payload(brands, timeframe="today 12-m", geo="")
            df = pytrends.interest_over_time()

            if df.empty:
                return None

            df = df.reset_index()

            value_cols = [b for b in brands if b in df.columns]
            df_tidy = df.melt(
                id_vars=["date", "isPartial"], 
                value_vars=value_cols, 
                var_name="brand", 
                value_name="value"
            )

            return df_tidy[["date", "value", "isPartial", "brand"]]

        except Exception as e:
            if "429" in str(e):
                wait = random.uniform(3, 9)
                print(f"429 detected, waiting {wait:.1f}s before retrying...")
                time.sleep(wait)
            else:
                print(f"Error: {e}")
                time.sleep(2)
    return None

We add another random delay between batches.

In [ ]:
all_results = []
counter = 0
batch_size = 5  

for i in range(0, len(brands_sorted), batch_size):
    batch = [b for b in brands_sorted[i:i + batch_size] if b not in done_brands]
    if not batch:
        continue

    print(f"Fetching batch: {batch}")
    df = fetch_trends_batch(batch)

    if df is not None:
        all_results.append(df)

        
        pd.concat([existing] + all_results, ignore_index=True).to_csv(res, index=False)
        existing = pd.read_csv(res)  
        all_results = [] 
    else:
        print(f"{batch}: no data or failed after retries")

    time.sleep(random.uniform(3, 7))

    counter += len(batch)
    if counter % 50 == 0:
        print(f"{counter} brands processed")

Fetching batch: ['360 Cashmere', 'Laundry by Shelli Segal', 'EDOX', 'HTC Los Angeles', 'Repeat']


/opt/anaconda3/envs/ecl-course-2025/lib/python3.11/site-packages/pytrends/request.py:260: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df = df.fillna(False)


Fetching batch: ['Part Two', 'VIA DELLE PERLE', 'NICE THINGS', 'Iris & Ink', 'JET SET']


/opt/anaconda3/envs/ecl-course-2025/lib/python3.11/site-packages/pytrends/request.py:260: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df = df.fillna(False)


Fetching batch: ['Alberto Biani', 'Eddy Monetti', 'Petit Bateau', 'Cinti', 'Princesse Tam Tam']


...

/opt/anaconda3/envs/ecl-course-2025/lib/python3.11/site-packages/pytrends/request.py:260: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df = df.fillna(False)


Fetching batch: ['Barrie', 'Aethera', 'Adidas x Craig Green', 'Lack Of Colour', 'Jack Wills']


/opt/anaconda3/envs/ecl-course-2025/lib/python3.11/site-packages/pytrends/request.py:260: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df = df.fillna(False)


Fetching batch: ['No faith studio', 'Finamore', 'Oxford', 'Sunflower', 'LES PETITES BOMBES']
429 detected, waiting 5.7s before retrying...
429 detected, waiting 7.3s before retrying...


KeyboardInterrupt: 

In [18]:
len(done_brands)

2195

The code was run repeatedly, IP bans occurred every 5 minutes after which, the IP address must be changed via using a VPN to rerun the code. 

In [19]:
df = pd.read_csv('google_trends_all_brands.csv')

df

,date,isPartial,brand,value
0,2024-11-24 00:00:00,False,Max Mara Weekend,91
1,2024-12-01 00:00:00,False,Max Mara Weekend,73
2,2024-12-08 00:00:00,False,Max Mara Weekend,75
3,2024-12-15 00:00:00,False,Max Mara Weekend,72
4,2024-12-22 00:00:00,False,Max Mara Weekend,90
...,...,...,...,...
116542,2025-10-26 00:00:00,False,Jack Wills,4
116543,2025-11-02 00:00:00,False,Jack Wills,4
116544,2025-11-09 00:00:00,False,Jack Wills,4
116545,2025-11-16 00:00:00,False,Jack Wills,5


In [20]:
df['value'] = df['value'].apply(float)

Compute for each brand, the mean popularity value.

In [30]:
res = df.groupby(['brand'],as_index=False)['value'].mean()

res

,brand,value
0,& Other Stories,62.811321
1,(+) PEOPLE,83.283019
2,032c,0.000000
3,10 Crosby by Derek Lam,0.000000
4,1017 ALYX 9SM,0.000000
...,...,...
2190,octopus,28.000000
2191,takeo kikuchi,4.018868
2192,valstar,0.000000
2193,yamamay,0.000000


In [31]:
res.to_csv('brand_popularity_mean.csv')